# Hourly Data

In [ ]:
import os
import time
import datetime
import pandas as pd
from binance.client import Client

# ================= CONFIG =================
coins = [
    'BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT',
    'ADAUSDT', 'DOGEUSDT', 'TRXUSDT', 'LINKUSDT', 'AVAXUSDT'
]

start_date_str = '01/01/2021'               # format: DD/MM/YYYY
end_date_str   = None                       # None = up to now
interval       = Client.KLINE_INTERVAL_1HOUR

output_dir = "binance_ohlcv_hourly"
os.makedirs(output_dir, exist_ok=True)

# ================= FETCH FUNCTION =================
def fetch_ohlcv(symbol, interval, start_str, end_str=None):
    """
    Fetch full OHLCV klines in chunks to handle large time ranges.
    Returns DataFrame with columns: open, high, low, close, volume
    """
    client = Client()  # No API key needed for public historical data

    start_ts = int(datetime.datetime.strptime(start_str, "%d/%m/%Y").timestamp() * 1000)
    if end_str is None:
        end_ts = int(time.time() * 1000)
    else:
        end_ts = int(datetime.datetime.strptime(end_str, "%d/%m/%Y").timestamp() * 1000)

    all_klines = []
    current_start = start_ts

    while current_start < end_ts:
        print(f"  Fetching {symbol} from {pd.to_datetime(current_start, unit='ms')} ...")
        try:
            klines = client.get_klines(
                symbol=symbol,
                interval=interval,
                startTime=current_start,
                limit=1000
            )
            if not klines:
                break
            all_klines.extend(klines)
            current_start = klines[-1][0] + 1  # next millisecond after last candle
            time.sleep(0.35)  # polite delay to avoid rate limits (~1200 req/min)
        except Exception as e:
            print(f"Error fetching {symbol}: {e}")
            time.sleep(10)  # wait longer on error
            continue

    if not all_klines:
        print(f"No data returned for {symbol}")
        return None

    df = pd.DataFrame(all_klines, columns=[
        'timestamp', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore'
    ])

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.set_index('timestamp')
    df = df[['open', 'high', 'low', 'close', 'volume']].astype(float)

    return df


# ================= MAIN =================
print(f"Starting data fetch for {len(coins)} coins")
print(f"Interval: {interval}")
print(f"From: {start_date_str}  →  To: {'now' if end_date_str is None else end_date_str}")
print(f"Saving to folder: {output_dir}\n")

for symbol in coins:
    print(f"\n=== {symbol} ===")
    df = fetch_ohlcv(symbol, interval, start_date_str, end_date_str)

    if df is not None and not df.empty:
        file_path = os.path.join(output_dir, f"{symbol}_ohlcv_1h.csv")
        df.to_csv(file_path)
        print(f"Saved {len(df)} rows → {file_path}")
        print(f"Date range: {df.index.min()} → {df.index.max()}")
    else:
        print(f"Failed to fetch or empty data for {symbol}")

print("\nData collection finished.")
print(f"All files are saved in: {os.path.abspath(output_dir)}")

# Daily Data

In [ ]:
import os
import time
import datetime
import pandas as pd
from binance.client import Client

# ================= CONFIG =================
# Same Top 20 as before (approximate Jan 2026 ranking)
coins = [
    # 01–10
    'BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT',
    'DOGEUSDT', 'ADAUSDT', 'TRXUSDT', 'AVAXUSDT', 'LINKUSDT',
    # 11–20
    'SUIUSDT', 'LTCUSDT', 'BCHUSDT', 'XLMUSDT', 'SHIBUSDT',
    'DOTUSDT', 'NEARUSDT', 'ATOMUSDT', 'MATICUSDT', 'HBARUSDT',
]

# ──────────────────────────────────────────────
# KEY CHANGE: very old start date → Binance will return from the earliest available
start_date_str = '01/08/2017'           # Safe old date (BTCUSDT starts ~Aug 2017)
# ──────────────────────────────────────────────

end_date_str   = None                   # → now
interval       = Client.KLINE_INTERVAL_1DAY

output_dir = "binance_ohlcv_daily_top20_fullhistory"
os.makedirs(output_dir, exist_ok=True)

# ================= FETCH FUNCTION =================
def fetch_ohlcv_daily(symbol, start_str, end_str=None):
    client = Client()  # public data only

    start_ts = int(datetime.datetime.strptime(start_str, "%d/%m/%Y").timestamp() * 1000)
    if end_str is None:
        end_ts = int(time.time() * 1000) + 86400000  # small buffer
    else:
        end_ts = int(datetime.datetime.strptime(end_str, "%d/%m/%Y").timestamp() * 1000)

    all_klines = []
    current_start = start_ts

    while current_start < end_ts:
        print(f"  Fetching {symbol} from {pd.to_datetime(current_start, unit='ms').date()} ...")
        try:
            klines = client.get_klines(
                symbol=symbol,
                interval=interval,
                startTime=current_start,
                limit=1000
            )
            if not klines:
                break

            all_klines.extend(klines)
            # Move to the millisecond after the last candle's close time
            current_start = klines[-1][6] + 1  # close_time + 1ms
            time.sleep(0.35)
        except Exception as e:
            if "invalid symbol" in str(e).lower():
                print(f"  → {symbol} may not exist or delisted")
                return None
            print(f"  Error: {e}")
            time.sleep(8)
            continue

    if not all_klines:
        print(f"  No data returned for {symbol}")
        return None

    df = pd.DataFrame(all_klines, columns=[
        'timestamp', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades', 'taker_buy_base',
        'taker_buy_quote', 'ignore'
    ])

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.set_index('timestamp')
    df = df[['open', 'high', 'low', 'close', 'volume']].astype(float)

    return df


# ================= MAIN =================
print(f"Fetching daily OHLCV → earliest available date for each coin")
print(f"(using dummy old start: {start_date_str} — Binance auto-adjusts)")
print(f"Up to: now")
print(f"Coins: {len(coins)}    Folder: {output_dir}\n")

for i, symbol in enumerate(coins, 1):
    rank = f"{i:02d}"
    print(f"\n=== {rank} | {symbol} ===")
    
    df = fetch_ohlcv_daily(symbol, start_date_str, end_date_str)

    if df is not None and not df.empty:
        file_path = os.path.join(output_dir, f"{symbol}_1d_full.csv")
        df.to_csv(file_path)
        days = len(df)
        start_date = df.index.min().date()
        end_date   = df.index.max().date()
        print(f"  Saved {days:,} rows  →  {file_path}")
        print(f"  Available range: {start_date}  →  {end_date}  ({days} days)")
        print(f"  Latest close: {df['close'].iloc[-1]:,.2f} USDT")
    else:
        print(f"  Failed / no data for {symbol}")

print("\n" + "="*70)
print("Done. Each coin now has data from its earliest available date on Binance spot.")
print(f"Files saved → {os.path.abspath(output_dir)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load your data
file_path = "/home/nckh2/qa/finance/full_result.csv"
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

# Pick any coin you want (change this line)
coin = 'BTCUSDT'   # or 'SOLUSDT', 'ETHUSDT', 'ADAUSDT', etc.

# Create figure with two y-axes
fig, ax1 = plt.subplots(figsize=(10, 5.5))

# Price line + simple candle-like fill
ax1.plot(df.index, df[f'{coin}_close'], color='#1f77b4', linewidth=1.4, label='Close Price')
ax1.set_ylabel('Price (USDT)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

# Green/red fill between open & close to fake candlestick feel
ax1.fill_between(df.index,
                 df[f'{coin}_open'], df[f'{coin}_close'],
                 where=(df[f'{coin}_close'] >= df[f'{coin}_open']),
                 color='green', alpha=0.25, interpolate=True)
ax1.fill_between(df.index,
                 df[f'{coin}_open'], df[f'{coin}_close'],
                 where=(df[f'{coin}_close'] < df[f'{coin}_open']),
                 color='red', alpha=0.25, interpolate=True)

# Volume on second axis
ax2 = ax1.twinx()
ax2.bar(df.index, df[f'{coin}_volume'], color='gray', alpha=0.5, width=0.9)
ax2.set_ylabel('Volume', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

# Nice date formatting
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate(rotation=45)

# Titles & layout
plt.title(f"{coin} – Price & Volume (simple static plot)")
fig.tight_layout()

# Save as PNG
output_file = f"simple_{coin}_plot.png"
plt.savefig(output_file, dpi=120, bbox_inches='tight')
plt.close()  # don't show in notebook if you don't want

print(f"Saved to: {output_file}")

In [ ]:
# all_models_forecast_merger_and_plotter_fixed.py
"""
Merge historical OHLCV data with 365-day forecasts from ALL models,
then generate plots comparing historical close vs each model's forecast.

Input files:
- Forecast: /home/nckh2/qa/all_models_crypto_forecast_365_days.csv (MultiIndex: model/date)
- Historical: /home/nckh2/qa/finance/binance_ohlcv_daily/*USDT_1d_full.csv

Output:
- Merged CSVs: /home/nckh2/qa/merged_all_models_historical_forecast/
- Plots:      /home/nckh2/qa/plots_all_models_forecast/
"""

import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import glob
from datetime import datetime

# ────────────────────────────────────────────────
# CONFIGURATION
# ────────────────────────────────────────────────
HISTORICAL_DIR    = "/home/nckh2/qa/finance/binance_ohlcv_daily"
FORECAST_CSV      = "/home/nckh2/qa/all_models_crypto_forecast_365_days.csv"
MERGED_DIR        = "/home/nckh2/qa/merged_all_models_historical_forecast"
PLOT_DIR          = "/home/nckh2/qa/plots_all_models_forecast"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Plotting style & appearance
plt.style.use('seaborn-v0_8-darkgrid')
FIGSIZE           = (14, 7)
DPI               = 140
SHOW_LAST_DAYS    = 180          # Historical days to display before forecast
EXPECTED_FORECAST = 365          # For reference / validation only

# Model-specific colors (expand as needed)
MODEL_COLORS = {
    'EnFormer':     '#1f77b4',
    'Autoformer':   '#ff7f0e',
    'FEDformer':    '#2ca02c',
    'Informer':     '#d62728',
    'iTransformer': '#9467bd',
    'RNN':          '#8c564b',
    'LSTM':         '#e377c2',
    'GRU':          '#7f7f7f',
    'CLAM':         '#bcbd22',
}

# ────────────────────────────────────────────────
# 1. Load the multi-model forecast file
# ────────────────────────────────────────────────
print("Loading forecast file...")
try:
    df_forecast = pd.read_csv(FORECAST_CSV, index_col=['model', 'date'])
    df_forecast.index = df_forecast.index.set_levels(
        pd.to_datetime(df_forecast.index.levels[1]), level='date'
    )
    df_forecast = df_forecast.sort_index(level='date')
except Exception as e:
    print(f"Error loading forecast CSV: {e}")
    exit(1)

models = df_forecast.index.get_level_values('model').unique().tolist()
coins  = df_forecast.columns.tolist()

print(f"→ Found {len(models)} models: {', '.join(models)}")
print(f"→ Coins: {', '.join(coins)}")
print(f"→ Forecast shape: {df_forecast.shape}\n")

# ────────────────────────────────────────────────
# 2. Discover historical files
# ────────────────────────────────────────────────
print("Scanning historical data files...")
pattern = os.path.join(HISTORICAL_DIR, "*USDT_1d_full.csv")
hist_files = glob.glob(pattern)

file_to_symbol = {}
for f in hist_files:
    stem = Path(f).stem
    symbol = stem.split('_')[0]  # e.g. BTCUSDT
    file_to_symbol[f] = symbol

print(f"→ Found {len(hist_files)} historical files\n")

# ────────────────────────────────────────────────
# 3. Merge historical + forecasts for each coin
# ────────────────────────────────────────────────
print("Merging historical data with forecasts...\n")

for hist_path, symbol in file_to_symbol.items():
    if symbol not in coins:
        print(f"Skipping {symbol} — not present in forecast file")
        continue

    print(f"Processing {symbol}...")

    # Load historical
    try:
        df_hist = pd.read_csv(
            hist_path,
            parse_dates=['timestamp'],
            usecols=['timestamp', 'open', 'high', 'low', 'close', 'volume']
        )
        df_hist = df_hist.set_index('timestamp').sort_index()
    except Exception as e:
        print(f"  Error reading historical file for {symbol}: {e}")
        continue

    # Extract all models' forecasts for this coin
    try:
        df_fc = df_forecast[symbol].unstack(level='model')
        df_fc.columns = [f'forecast_close_{m}' for m in df_fc.columns]
    except Exception as e:
        print(f"  Error extracting forecasts for {symbol}: {e}")
        continue

    # Merge (outer join → keeps all history + future forecasts)
    df_merged = df_hist.join(df_fc, how='outer')

    # Save merged file
    out_filename = f"{symbol}_historical_plus_all_forecasts.csv"
    out_path = os.path.join(MERGED_DIR, out_filename)

    df_merged.to_csv(out_path, date_format='%Y-%m-%d')

    forecast_len = len(df_fc) if not df_fc.empty else 0
    print(f"  → Saved merged CSV: {out_path}")
    print(f"     Shape: {df_merged.shape} | Historic rows: {len(df_hist)} | Forecast rows/model: {forecast_len}")
    if forecast_len != EXPECTED_FORECAST:
        print(f"     WARNING: Forecast length {forecast_len} ≠ expected {EXPECTED_FORECAST}")

# ────────────────────────────────────────────────
# 4. Generate plots from merged files
# ────────────────────────────────────────────────
print("\nGenerating plots...\n")

merged_pattern = os.path.join(MERGED_DIR, "*_historical_plus_all_forecasts.csv")
merged_files = sorted(glob.glob(merged_pattern))

if not merged_files:
    print(f"No merged files found in {MERGED_DIR}")
else:
    print(f"Found {len(merged_files)} merged files to plot\n")

    for filepath in merged_files:
        coin = Path(filepath).stem.split('_')[0]
        print(f"Plotting {coin}...")

        try:
            df = pd.read_csv(filepath, parse_dates=['timestamp'], index_col='timestamp')
            df = df.sort_index()
        except Exception as e:
            print(f"  Error reading {filepath}: {e}")
            continue

        if 'close' not in df.columns:
            print(f"  No 'close' column found — skipping")
            continue

        real_close = df['close']

        forecast_cols = [c for c in df.columns if c.startswith('forecast_close_')]
        if not forecast_cols:
            print(f"  No forecast columns found — skipping")
            continue

        # Determine plot start date
        hist_valid = real_close.dropna()
        if len(hist_valid) > SHOW_LAST_DAYS:
            plot_start = hist_valid.index[-SHOW_LAST_DAYS]
        else:
            plot_start = hist_valid.index[0]

        df_plot = df.loc[plot_start:]

        last_hist_date = hist_valid.index[-1]

        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

        # Historical close (with outline for visibility)
        ax.plot(df_plot.index, df_plot['close'],
                color='white', linewidth=2.8, zorder=10, label='Historical Close')
        ax.plot(df_plot.index, df_plot['close'],
                color='#444444', linewidth=1.2, alpha=0.95, zorder=11)

        # Each model's forecast
        for fc_col in forecast_cols:
            model = fc_col.replace('forecast_close_', '')
            pred = df_plot[fc_col]

            # Only plot forecast part
            pred = pred[pred.index >= last_hist_date + pd.Timedelta(days=1)]
            if pred.empty:
                continue

            color = MODEL_COLORS.get(model, '#aaaaaa')

            ax.plot(pred.index, pred,
                    color=color, linewidth=2.1, alpha=0.88,
                    label=f"{model} forecast")

            # Vertical line at start of this forecast
            if not pred.empty:
                ax.axvline(pred.index[0], color=color, linestyle='--', alpha=0.45,
                           linewidth=1.1, zorder=5)

        # Vertical separator at forecast start
        ax.axvline(last_hist_date, color='0.3', linestyle='--', linewidth=1.8,
                   alpha=0.75, label='Forecast start')

        # Styling
        ax.set_title(f"{coin} — Historical vs 365-day Forecasts (all models)",
                     fontsize=16, pad=16)
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel("Close Price (USDT)", fontsize=12)

        ax.legend(loc='upper left', fontsize=10, ncol=2, framealpha=0.93)
        ax.grid(True, alpha=0.3, linestyle='--')
        plt.xticks(rotation=30)
        plt.tight_layout()

        # Save plot
        safe_coin = coin.replace('/', '_')
        plot_path = os.path.join(PLOT_DIR, f"{safe_coin}_all_models_forecast.png")
        plt.savefig(plot_path, dpi=180, bbox_inches='tight')
        plt.close(fig)

        print(f"  → Saved plot: {plot_path}")

print("\n" + "="*70)
print("Finished!")
print(f"Merged files → {MERGED_DIR}")
print(f"Plots        → {PLOT_DIR}")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import glob

# ────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────
INPUT_DIR   = "/home/nckh2/qa/merged_historical_forecast"
OUTPUT_DIR  = "/home/nckh2/qa/plots_historical_vs_forecast"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.style.use('ggplot')          # nice look (you can change to 'default' or remove)
FIGSIZE     = (12, 6)
DPI         = 120

# ────────────────────────────────────────────────
# Find all merged files
# ────────────────────────────────────────────────
pattern = os.path.join(INPUT_DIR, "*_historical_plus_forecast.csv")
files = sorted(glob.glob(pattern))

if not files:
    print("No files found in", INPUT_DIR)
    exit()

print(f"Found {len(files)} coins to plot\n")

for filepath in files:
    coin = Path(filepath).stem.split('_')[0]  # e.g. ADAUSDT
    print(f"Processing {coin} ...")

    # Read data
    df = pd.read_csv(filepath, parse_dates=['timestamp'], index_col='timestamp')
    df = df.sort_index()

    # Extract the two series we care about
    if 'close' in df.columns:
        real = df['close']
    elif 'historical_close' in df.columns:
        real = df['historical_close']
    else:
        print(f"  ⚠️ No 'close' or 'historical_close' column in {coin}")
        continue

    if 'forecast_close' in df.columns:
        pred = df['forecast_close']
    else:
        print(f"  ⚠️ No 'forecast_close' column in {coin}")
        continue

    # Create figure
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Plot real (historical) — blue solid line
    ax.plot(real.index, real, color='royalblue', linewidth=1.8,
            label='Historical Close')

    # Plot predicted (forecast) — red line, only where we have values
    ax.plot(pred.index, pred, color='crimson', linewidth=2.0,
            label='Forecast Close')

    # Vertical line at the switch point (last real value)
    if len(real.dropna()) > 0:
        last_real_date = real.dropna().index[-1]
        ax.axvline(x=last_real_date, color='0.4', linestyle='--', alpha=0.7,
                   linewidth=1.2, label='Forecast starts')

    # Formatting
    ax.set_title(f"{coin} — Historical vs Forecast Close Price", fontsize=16, pad=12)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Price (USDT)", fontsize=12)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

    # Rotate x-ticks for readability
    plt.xticks(rotation=45)

    # Tight layout & save
    plt.tight_layout()
    out_png = os.path.join(OUTPUT_DIR, f"{coin}_hist_vs_forecast.png")
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close(fig)   # important — free memory

    print(f"  Saved: {out_png}")

print("\nAll plots saved in:", OUTPUT_DIR)

In [ ]:
# all_models_forecast_merger_and_plotter_fixed.py
"""
Merge historical OHLCV data with 365-day forecasts from ALL models,
then generate plots comparing historical close vs each model's forecast.

Input files:
- Forecast: /home/nckh2/qa/all_models_crypto_forecast_365_days.csv (MultiIndex: model/date)
- Historical: /home/nckh2/qa/finance/binance_ohlcv_daily/*USDT_1d_full.csv

Output:
- Merged CSVs: /home/nckh2/qa/merged_all_models_historical_forecast/
- Plots:      /home/nckh2/qa/plots_all_models_forecast/
"""

import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import glob
from datetime import datetime

# ────────────────────────────────────────────────
# CONFIGURATION
# ────────────────────────────────────────────────
HISTORICAL_DIR    = "/home/nckh2/qa/finance/binance_ohlcv_daily"
FORECAST_CSV      = "/home/nckh2/qa/all_models_crypto_forecast_365_days.csv"
MERGED_DIR        = "/home/nckh2/qa/merged_all_models_historical_forecast"
PLOT_DIR          = "/home/nckh2/qa/plots_all_models_forecast"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Plotting style & appearance
plt.style.use('seaborn-v0_8-darkgrid')
FIGSIZE           = (14, 7)
DPI               = 140
SHOW_LAST_DAYS    = 180          # Historical days to display before forecast
EXPECTED_FORECAST = 365          # For reference / validation only

# Model-specific colors (expand as needed)
MODEL_COLORS = {
    'EnFormer':     '#1f77b4',
    'Autoformer':   '#ff7f0e',
    'FEDformer':    '#2ca02c',
    'Informer':     '#d62728',
    'iTransformer': '#9467bd',
    'RNN':          '#8c564b',
    'LSTM':         '#e377c2',
    'GRU':          '#7f7f7f',
    'CLAM':         '#bcbd22',
}

# ────────────────────────────────────────────────
# 1. Load the multi-model forecast file
# ────────────────────────────────────────────────
print("Loading forecast file...")
try:
    df_forecast = pd.read_csv(FORECAST_CSV, index_col=['model', 'date'])
    df_forecast.index = df_forecast.index.set_levels(
        pd.to_datetime(df_forecast.index.levels[1]), level='date'
    )
    df_forecast = df_forecast.sort_index(level='date')
except Exception as e:
    print(f"Error loading forecast CSV: {e}")
    exit(1)

models = df_forecast.index.get_level_values('model').unique().tolist()
coins  = df_forecast.columns.tolist()

print(f"→ Found {len(models)} models: {', '.join(models)}")
print(f"→ Coins: {', '.join(coins)}")
print(f"→ Forecast shape: {df_forecast.shape}\n")

# ────────────────────────────────────────────────
# 2. Discover historical files
# ────────────────────────────────────────────────
print("Scanning historical data files...")
pattern = os.path.join(HISTORICAL_DIR, "*USDT_1d_full.csv")
hist_files = glob.glob(pattern)

file_to_symbol = {}
for f in hist_files:
    stem = Path(f).stem
    symbol = stem.split('_')[0]  # e.g. BTCUSDT
    file_to_symbol[f] = symbol

print(f"→ Found {len(hist_files)} historical files\n")

# ────────────────────────────────────────────────
# 3. Merge historical + forecasts for each coin
# ────────────────────────────────────────────────
print("Merging historical data with forecasts...\n")

for hist_path, symbol in file_to_symbol.items():
    if symbol not in coins:
        print(f"Skipping {symbol} — not present in forecast file")
        continue

    print(f"Processing {symbol}...")

    # Load historical
    try:
        df_hist = pd.read_csv(
            hist_path,
            parse_dates=['timestamp'],
            usecols=['timestamp', 'open', 'high', 'low', 'close', 'volume']
        )
        df_hist = df_hist.set_index('timestamp').sort_index()
    except Exception as e:
        print(f"  Error reading historical file for {symbol}: {e}")
        continue

    # Extract all models' forecasts for this coin
    try:
        df_fc = df_forecast[symbol].unstack(level='model')
        df_fc.columns = [f'forecast_close_{m}' for m in df_fc.columns]
    except Exception as e:
        print(f"  Error extracting forecasts for {symbol}: {e}")
        continue

    # Merge (outer join → keeps all history + future forecasts)
    df_merged = df_hist.join(df_fc, how='outer')

    # Save merged file
    out_filename = f"{symbol}_historical_plus_all_forecasts.csv"
    out_path = os.path.join(MERGED_DIR, out_filename)

    df_merged.to_csv(out_path, date_format='%Y-%m-%d')

    forecast_len = len(df_fc) if not df_fc.empty else 0
    print(f"  → Saved merged CSV: {out_path}")
    print(f"     Shape: {df_merged.shape} | Historic rows: {len(df_hist)} | Forecast rows/model: {forecast_len}")
    if forecast_len != EXPECTED_FORECAST:
        print(f"     WARNING: Forecast length {forecast_len} ≠ expected {EXPECTED_FORECAST}")



In [ ]:
import pandas as pd

# Path to the forecast CSV
forecast_path = '/home/nckh2/qa/all_models_crypto_forecast_365_days.csv'

# Read the forecast data
df_forecast = pd.read_csv(forecast_path)

# Melt the forecast to long format: model, timestamp, coin, close
id_vars = ['model', 'timestamp']
value_vars = [col for col in df_forecast.columns if col not in id_vars]
df_forecast_long = pd.melt(df_forecast, id_vars=id_vars, value_vars=value_vars, var_name='coin', value_name='close')

# Directory for historical OHLCV CSVs
historical_dir = '/home/nckh2/qa/finance/binance_ohlcv_daily/'

# List of coins from forecast
coins = value_vars

# Read historical data for each coin and convert to long format
df_hist_list = []
for coin in coins:
    hist_path = f'{historical_dir}{coin}_1d_full.csv'
    df_hist = pd.read_csv(hist_path)
    df_hist = df_hist[['timestamp', 'close']]
    df_hist['coin'] = coin
    df_hist['model'] = 'historical'
    df_hist_list.append(df_hist)

df_hist_long = pd.concat(df_hist_list)

# Combine historical and forecast data
df_combined = pd.concat([df_hist_long, df_forecast_long])

# Convert timestamp to datetime for sorting
df_combined['timestamp'] = pd.to_datetime(df_combined['timestamp'])

# Sort by timestamp, coin, model
df_combined = df_combined.sort_values(['timestamp', 'coin', 'model'])

# Output path for the combined CSV
output_path = '/home/nckh2/qa/combined_crypto_prices.csv'

# Save to CSV
df_combined.to_csv(output_path, index=False)

print(f"Combined data saved to {output_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

# Path to the combined CSV
combined_path = '/home/nckh2/qa/combined_crypto_prices.csv'

# Read the combined data
df = pd.read_csv(combined_path)

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Get unique coins
coins = df['coin'].unique()

# Models (excluding historical)
models = ['EnFormer', 'iTransformer', 'Fedformer', 'Informer', 'RNN', 'LSTM', 'GRU', 'CLAM']

for coin in coins:
    # Filter data for the coin
    df_coin = df[df['coin'] == coin].copy()
    
    # Historical data
    df_hist = df_coin[df_coin['model'] == 'historical']
    
    # If no historical data, skip
    if df_hist.empty:
        continue
    
    # Calculate reasonable y-limits based on historical data
    hist_min = df_hist['close'].min()
    hist_max = df_hist['close'].max()
    y_min = max(0, hist_min * 0.9)
    y_max = hist_max * 2.0  # Double the historical max to allow some growth visibility
    
    # Create plot
    plt.figure(figsize=(12, 6))
    
    # Plot historical
    plt.plot(df_hist['timestamp'], df_hist['close'], label='Historical', color='black', linewidth=2)
    
    # Plot each model's forecast
    for model in models:
        df_model = df_coin[df_coin['model'] == model]
        if not df_model.empty:
            plt.plot(df_model['timestamp'], df_model['close'], label=model, linestyle='--', alpha=0.7)
    
    # Set limits
    plt.ylim(y_min, y_max)
    
    # Formatting
    plt.xlabel('Timestamp')
    plt.ylabel('Close Price')
    plt.title(f'{coin} Close Prices: Historical and Forecasts')
    plt.legend()
    plt.grid(True)
    plt.gca().xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    
    # Save plot
    plot_path = f'/home/nckh2/qa/{coin}_plot.png'
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.close()
    
    print(f"Plot saved for {coin} at {plot_path}")

In [ ]:
import pandas as pd
import os
from pathlib import Path
import glob

# ────────────────────────────────────────────────
#  CONFIG
# ────────────────────────────────────────────────
HISTORICAL_DIR = "/home/nckh2/qa/finance/binance_ohlcv_daily"
FORECAST_FILE  = "/home/nckh2/qa/crypto_forecast_365_days.csv"
OUTPUT_DIR     = "/home/nckh2/qa/merged_historical_forecast"   # ← change if you want

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ────────────────────────────────────────────────
#  1. Read forecast file (wide format)
# ────────────────────────────────────────────────
df_forecast = pd.read_csv(FORECAST_FILE)

# Standardize column names
df_forecast = df_forecast.rename(columns={
    'Timestamp': 'timestamp'
})

# Convert to datetime and set index
df_forecast['timestamp'] = pd.to_datetime(df_forecast['timestamp'])
df_forecast.set_index('timestamp', inplace=True)

# Available coins in forecast
forecast_coins = [col for col in df_forecast.columns if col != 'timestamp']
print("Coins found in forecast file:", forecast_coins)

# ────────────────────────────────────────────────
#  2. Find all historical files
# ────────────────────────────────────────────────
pattern = os.path.join(HISTORICAL_DIR, "*USDT_1d_full.csv")
historical_files = glob.glob(pattern)

# Extract symbol from filename
file_to_symbol = {}
for f in historical_files:
    stem = Path(f).stem
    symbol = stem.split('_')[0]          # ADAUSDT, BTCUSDT, ...
    file_to_symbol[f] = symbol

print(f"Found {len(historical_files)} historical files")

# ────────────────────────────────────────────────
#  3. Process each coin that exists in BOTH sources
# ────────────────────────────────────────────────
for hist_path, symbol in file_to_symbol.items():
    if symbol not in forecast_coins:
        print(f"Skipping {symbol} — not found in forecast file")
        continue

    print(f"\nProcessing {symbol} ...")

    # Read historical data
    df_hist = pd.read_csv(hist_path,
                          parse_dates=['timestamp'],
                          usecols=['timestamp', 'open', 'high', 'low', 'close'])

    df_hist.set_index('timestamp', inplace=True)
    df_hist = df_hist.sort_index()   # just in case

    # Get forecast series for this coin
    df_fc_col = df_forecast[[symbol]].copy()
    df_fc_col = df_fc_col.rename(columns={symbol: 'forecast_close'})

    # ── Merge ─────────────────────────────────────
    # Historical → forecast (keep all historical + future forecast)
    df_merged = df_hist.join(df_fc_col, how='outer')

    # Optional: fill NaN in historical columns if you want forward-fill etc.
    # df_merged[['open','high','low','close','volume']] = df_merged[['open','high','low','close','volume']].ffill()

    # Make sure index is datetime
    df_merged.index.name = 'timestamp'

    # ── Save ──────────────────────────────────────
    out_filename = f"{symbol}_historical_plus_forecast.csv"
    out_path = os.path.join(OUTPUT_DIR, out_filename)
    
    df_merged.to_csv(out_path,
                     date_format='%Y-%m-%d')   # or '%Y-%m-%d %H:%M:%S' if needed

    print(f"  Saved → {out_path}")
    print(f"  Shape: {df_merged.shape}  |  Historic rows: {len(df_hist)}  |  Forecast rows: {len(df_fc_col)}")

print("\nDone.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import glob

# ────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────
INPUT_DIR   = "/home/nckh2/qa/merged_historical_forecast"
OUTPUT_DIR  = "/home/nckh2/qa/plots_historical_vs_forecast"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.style.use('ggplot')          # nice look (you can change to 'default' or remove)
FIGSIZE     = (12, 6)
DPI         = 300

# ────────────────────────────────────────────────
# Find all merged files
# ────────────────────────────────────────────────
pattern = os.path.join(INPUT_DIR, "*_historical_plus_forecast.csv")
files = sorted(glob.glob(pattern))

if not files:
    print("No files found in", INPUT_DIR)
    exit()

print(f"Found {len(files)} coins to plot\n")

for filepath in files:
    coin = Path(filepath).stem.split('_')[0]  # e.g. ADAUSDT
    print(f"Processing {coin} ...")

    # Read data
    df = pd.read_csv(filepath, parse_dates=['timestamp'], index_col='timestamp')
    df = df.sort_index()

    # Extract the two series we care about
    if 'close' in df.columns:
        real = df['close']
    elif 'historical_close' in df.columns:
        real = df['historical_close']
    else:
        print(f"  ⚠️ No 'close' or 'historical_close' column in {coin}")
        continue

    if 'forecast_close' in df.columns:
        pred = df['forecast_close']
    else:
        print(f"  ⚠️ No 'forecast_close' column in {coin}")
        continue

    # Create figure
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Plot real (historical) — blue solid line
    ax.plot(real.index, real, color='royalblue', linewidth=1.8,
            label='Historical Close')

    # Plot predicted (forecast) — red line, only where we have values
    ax.plot(pred.index, pred, color='crimson', linewidth=2.0,
            label='Forecast Close')

    # Vertical line at the switch point (last real value)
    if len(real.dropna()) > 0:
        last_real_date = real.dropna().index[-1]
        ax.axvline(x=last_real_date, color='0.4', linestyle='--', alpha=0.7,
                   linewidth=1.2, label='Forecast starts')

    # Formatting
    ax.set_title(f"{coin} — Historical vs Forecast Close Price", fontsize=16, pad=12)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Price (USDT)", fontsize=12)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

    # Rotate x-ticks for readability
    plt.xticks(rotation=45)

    # Tight layout & save
    plt.tight_layout()
    out_png = os.path.join(OUTPUT_DIR, f"{coin}_hist_vs_forecast.png")
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close(fig)   # important — free memory

    print(f"  Saved: {out_png}")

print("\nAll plots saved in:", OUTPUT_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# ────────────────────────────────────────────────
# Config – change these as needed
# ────────────────────────────────────────────────

BASE_DIR = Path("/home/nckh2/qa")
COINS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ADAUSDT', 'XRPUSDT']  # pick your favorites
MODELS = ['RNN', 'LSTM', 'GRU', 'CLAM']                         # which model forecasts you have

PLOT_ONE_STEP       = True
PLOT_AUTOREGRESSIVE = True
PLOT_MULTIPLE_MODELS = True     # compare models on same coin
PLOT_LOG_SCALE      = False     # useful for crypto
SHOW_GRID           = True
FIGSIZE             = (14, 7)

# ────────────────────────────────────────────────
# Helper – load one forecast file
# ────────────────────────────────────────────────

def load_forecast(model_name: str) -> pd.DataFrame:
    path = BASE_DIR / f"forecasts_{model_name}_full_ohlcv.csv"
    if not path.exists():
        print(f"File not found: {path}")
        return None
    
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = 'date'
    return df

# ────────────────────────────────────────────────
# Variant 1: Plot one coin – one model – both prediction types
# ────────────────────────────────────────────────

def plot_one_model_one_coin(model_name: str, coin: str):
    df = load_forecast(model_name)
    if df is None:
        return
    
    true_col   = f"{coin}_true"
    one_col    = f"{coin}_one_step"
    autoreg_col = f"{coin}_autoregressive"
    
    if true_col not in df.columns:
        print(f"Column {true_col} not found in {model_name} data")
        return
    
    plt.figure(figsize=FIGSIZE)
    
    plt.plot(df.index, df[true_col], 
             label='Actual', color='black', linewidth=1.4, zorder=3)
    
    if PLOT_ONE_STEP and one_col in df.columns:
        plt.plot(df.index, df[one_col], 
                 label='One-step ahead', color='#1f77b4', alpha=0.9)
    
    if PLOT_AUTOREGRESSIVE and autoreg_col in df.columns:
        plt.plot(df.index, df[autoreg_col], 
                 label='Autoregressive (recursive)', 
                 color='#ff7f0e', linestyle='--', linewidth=1.8)
    
    title = f"{coin} – {model_name}  (test period)"
    if PLOT_LOG_SCALE:
        plt.yscale('log')
        title += "  (log scale)"
    
    plt.title(title, fontsize=14, pad=12)
    plt.xlabel("Date")
    plt.ylabel("Close Price (USDT)")
    plt.legend(loc='upper left')
    if SHOW_GRID:
        plt.grid(True, alpha=0.3, linestyle=':')
    
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=30)
    
    plt.tight_layout()
    plt.show()

# ────────────────────────────────────────────────
# Variant 2: Compare multiple models on the same coin
# ────────────────────────────────────────────────

def plot_compare_models_on_coin(coin: str, models_to_compare=None):
    if models_to_compare is None:
        models_to_compare = MODELS
    
    plt.figure(figsize=(16, 8))
    
    # Plot true once
    df_ref = None
    for m in models_to_compare:
        df = load_forecast(m)
        if df is None:
            continue
        true_col = f"{coin}_true"
        if true_col in df.columns and df_ref is None:
            df_ref = df
            plt.plot(df.index, df[true_col], 
                     label='Actual', color='black', linewidth=1.6, zorder=10)
    
    if df_ref is None:
        print(f"No true price data found for {coin}")
        return
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    for i, m in enumerate(models_to_compare):
        df = load_forecast(m)
        if df is None:
            continue
        autoreg_col = f"{coin}_autoregressive"
        if autoreg_col in df.columns:
            plt.plot(df.index, df[autoreg_col],
                     label=f'{m} (autoregressive)',
                     color=colors[i % len(colors)], linestyle='--', alpha=0.9)
        
        one_col = f"{coin}_one_step"
        if PLOT_ONE_STEP and one_col in df.columns:
            plt.plot(df.index, df[one_col],
                     label=f'{m} (one-step)',
                     color=colors[i % len(colors)], alpha=0.7)
    
    title = f"{coin} – Model Comparison (test set)"
    if PLOT_LOG_SCALE:
        plt.yscale('log')
        title += " – log scale"
    
    plt.title(title, fontsize=15, pad=14)
    plt.xlabel("Date")
    plt.ylabel("Close Price (USDT)")
    plt.legend(loc='upper left', ncol=2, fontsize=9.5)
    if SHOW_GRID:
        plt.grid(True, alpha=0.25, linestyle=':')
    
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=30)
    
    plt.tight_layout()
    plt.show()

# ────────────────────────────────────────────────
# Examples – uncomment what you want to see
# ────────────────────────────────────────────────

# Single model + single coin
# plot_one_model_one_coin('CLAM', 'BTCUSDT')
# plot_one_model_one_coin('LSTM', 'SOLUSDT')
# plot_one_model_one_coin('GRU', 'ETHUSDT')

# Compare models on one coin (recommended)
plot_compare_models_on_coin('BTCUSDT')
# plot_compare_models_on_coin('ETHUSDT')
# plot_compare_models_on_coin('SOLUSDT', models_to_compare=['LSTM', 'GRU', 'CLAM'])

print("Plotting done.")

In [ ]:
import pandas as pd
import openpyxl
from pathlib import Path

# ────────────────────────────────────────────────
# Paths
# ────────────────────────────────────────────────

BASE_DIR = Path("/home/nckh2/qa")

GLOBAL_FILES = [
    'one_step_results_global.csv',
    'one_step_EnFormer_global.csv',
    'one_step_transformer_global.csv',
]

PER_COIN_FILES = [
    'one_step_results_per_coin.csv',
    'one_step_EnFormer_per_coin.csv',
    'one_step_transformer_per_coin.csv',
]

OUTPUT_FILE = BASE_DIR / "model_comparison_summary.csv"

# ────────────────────────────────────────────────
# Load and combine global results
# ────────────────────────────────────────────────

global_dfs = []
for f in GLOBAL_FILES:
    path = BASE_DIR / f
    if path.exists():
        df = pd.read_csv(path)
        global_dfs.append(df)

if global_dfs:
    global_df = pd.concat(global_dfs, ignore_index=True)
    global_df = global_df.sort_values('test_rmse_global').reset_index(drop=True)
    print("\nGlobal RMSE Ranking (scaled space):")
    print(global_df[['model', 'test_rmse_global', 'test_mae_global']].round(6))
else:
    print("No global results files found.")
    global_df = pd.DataFrame()

# ────────────────────────────────────────────────
# Load and combine per-coin results
# ────────────────────────────────────────────────

per_coin_dfs = []
for f in PER_COIN_FILES:
    path = BASE_DIR / f
    if path.exists():
        df = pd.read_csv(path)
        per_coin_dfs.append(df)

if per_coin_dfs:
    per_coin_df = pd.concat(per_coin_dfs, ignore_index=True)
    
    # Pivot to wide format: models as columns, coins as rows
    pivot_rmse = per_coin_df.pivot_table(
        index='coin',
        columns='model',
        values='rmse_scaled',
        aggfunc='first'
    ).round(6)
    
    pivot_mae = per_coin_df.pivot_table(
        index='coin',
        columns='model',
        values='mae_scaled',
        aggfunc='first'
    ).round(6)
    
    print("\nPer-coin RMSE (scaled space) — lower is better:")
    print(pivot_rmse)
    
    print("\nPer-coin MAE (scaled space):")
    print(pivot_mae)
    
    # Optional: add average per model
    pivot_rmse.loc['Average'] = pivot_rmse.mean()
    pivot_mae.loc['Average'] = pivot_mae.mean()
    
    # Save combined comparison
    with pd.ExcelWriter(OUTPUT_FILE.with_suffix('.xlsx')) as writer:
        global_df.to_excel(writer, sheet_name='Global', index=False)
        pivot_rmse.to_excel(writer, sheet_name='RMSE_per_coin')
        pivot_mae.to_excel(writer, sheet_name='MAE_per_coin')
    
    print(f"\nComparison table saved to: {OUTPUT_FILE.with_suffix('.xlsx')}")
else:
    print("No per-coin results files found.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Patch
import numpy as np
import os

# ────────────────────────────────────────────────
#  Configuration
# ────────────────────────────────────────────────

FC_PATH = "/home/nckh2/qa/forecasts_EnFormer_full_ohlcv.csv"
COIN = "BTCUSDT"

FIGSIZE = (26, 11)
ZOOM_LAST_DAYS = 180        # None = full history
# ZOOM_LAST_DAYS = None

CANDLE_WIDTH = 0.48
OFFSET = 0.26

# ──── Highlight settings ───────────────────────────────────────
WINDOW_DAYS = 7
MAPE_THRESHOLD = 3.0       # % — lower = stricter (try 3.0–6.0)
HIGHLIGHT_ALPHA = 0.09     # very light / subtle grey
# ────────────────────────────────────────────────────────────────

# Output directory and filename
SAVE_DIR = "/home/nckh2/qa/finance/plots"  # ← change if needed
os.makedirs(SAVE_DIR, exist_ok=True)

zoom_str = f"_zoom{ZOOM_LAST_DAYS}d" if ZOOM_LAST_DAYS else "_full"
OUTPUT_FILENAME = os.path.join(SAVE_DIR, f"{COIN}_real_vs_pred{zoom_str}_600dpi.png")

# ────────────────────────────────────────────────
#  Load & prepare data
# ────────────────────────────────────────────────

df = pd.read_csv(FC_PATH)
df.columns = df.columns.str.lower().str.strip()

coin_prefix = f"{COIN.lower()}_"

true_o = f"{coin_prefix}open_true"
true_h = f"{coin_prefix}high_true"
true_l = f"{coin_prefix}low_true"
true_c = f"{coin_prefix}close_true"

pred_o = f"{coin_prefix}open_one_step"
pred_h = f"{coin_prefix}high_one_step"
pred_l = f"{coin_prefix}low_one_step"
pred_c = f"{coin_prefix}close_one_step"

required_true = [true_o, true_h, true_l, true_c]
required_pred = [pred_o, pred_h, pred_l, pred_c]

missing_true = [col for col in required_true if col not in df.columns]
missing_pred = [col for col in required_pred if col not in df.columns]

if missing_true:
    raise ValueError(f"Missing true columns: {missing_true}")

HAS_FULL_PRED = len(missing_pred) == 0

if not HAS_FULL_PRED:
    print("No full predicted OHLC → highlighting disabled")
    HIGHLIGHT = False
else:
    HIGHLIGHT = True

cols = ['timestamp'] + required_true
if HAS_FULL_PRED:
    cols += required_pred

df = df[cols].copy()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')

if ZOOM_LAST_DAYS is not None:
    zoom_start = df.index.max() - pd.Timedelta(days=ZOOM_LAST_DAYS + 60)
    df_plot = df.loc[df.index >= zoom_start].copy()
else:
    df_plot = df.copy()

# ──── Find non-overlapping good windows ────────────────────────
highlight_spans = []

if HIGHLIGHT:
    closes_real = df_plot[true_c]
    closes_pred = df_plot[pred_c]
    
    n = len(df_plot)
    step = WINDOW_DAYS
    
    for start_idx in range(0, n, step):
        end_idx = min(start_idx + WINDOW_DAYS, n)
        if end_idx - start_idx < WINDOW_DAYS // 2:  # skip tiny last chunk
            break
            
        window_real = closes_real.iloc[start_idx:end_idx]
        window_pred = closes_pred.iloc[start_idx:end_idx]
        
        if len(window_real) < 2:
            continue
            
        norm_factor = window_real.iloc[0]
        if norm_factor <= 0:
            continue
            
        rel_real = window_real / norm_factor
        rel_pred = window_pred / norm_factor
        
        mape = np.mean(np.abs((rel_real - rel_pred) / rel_real)) * 100
        
        if mape <= MAPE_THRESHOLD:
            start_time = window_real.index[0]
            end_time   = window_real.index[-1]
            # Extend end a tiny bit for visual connection if next is also good
            if end_idx < n:
                end_time = df_plot.index[end_idx]  # reach start of next block
            highlight_spans.append((start_time, end_time))

print(f"Found {len(highlight_spans)} good 7-day blocks (MAPE ≤ {MAPE_THRESHOLD}%)")

# ────────────────────────────────────────────────
#  Plotting
# ────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=100)  # working dpi (screen)

def plot_candles(ax, data, o_col, h_col, l_col, c_col,
                 colorup, colordown,
                 edgecolor='black', wickcolor='black',
                 width=CANDLE_WIDTH, alpha=1.0,
                 offset=0.0,
                 hollow=False,
                 zorder_body=3, zorder_wick=2):
    
    for dt, row in data.iterrows():
        x = mdates.date2num(dt) + offset
        o = row[o_col]
        h = row[h_col]
        l = row[l_col]
        c = row[c_col]

        body_color = colorup if c >= o else colordown

        if hollow:
            facecolor = 'none'
            ec = body_color
            lw_body = 2.0
        else:
            facecolor = body_color
            ec = edgecolor
            lw_body = 1.1

        bottom = min(o, c)
        height = abs(c - o)

        ax.add_patch(Rectangle(
            (x - width/2, bottom), width, height,
            facecolor=facecolor,
            edgecolor=ec,
            linewidth=lw_body,
            alpha=alpha,
            zorder=zorder_body
        ))

        ax.plot([x, x], [l, h],
                color=wickcolor,
                linewidth=2.1 if hollow else 1.8,
                alpha=alpha,
                zorder=zorder_wick)

        if hollow:
            ax.plot([x, x], [o, c],
                    color='black',
                    linewidth=2.3,
                    alpha=alpha,
                    zorder=zorder_body + 1)


# Real candles
plot_candles(ax, df_plot,
             true_o, true_h, true_l, true_c,
             colorup='#00c853', colordown='#d50000',
             edgecolor='black', wickcolor='#455a64',
             width=CANDLE_WIDTH, alpha=1.0,
             offset=-OFFSET,
             hollow=True,
             zorder_body=3, zorder_wick=2)

# Predicted candles
if HAS_FULL_PRED:
    plot_candles(ax, df_plot,
                 pred_o, pred_h, pred_l, pred_c,
                 colorup='#ffca28', colordown='black',
                 edgecolor='black', wickcolor='#212121',
                 width=CANDLE_WIDTH, alpha=0.92,
                 offset=+OFFSET,
                 hollow=False,
                 zorder_body=5, zorder_wick=4.5)

# ── Add light grey background to good blocks ───────────────────
for start, end in highlight_spans:
    ax.axvspan(start, end,
               facecolor='gray', alpha=HIGHLIGHT_ALPHA,
               zorder=1, linewidth=0)

# ── Styling ────────────────────────────────────────────────────
zoom_text = f" — last ~{ZOOM_LAST_DAYS} days" if ZOOM_LAST_DAYS else ""
# title = f"{COIN}   Real (left, outline) vs Predicted (right, filled){zoom_text}\n"
# title += f"Light grey = good match periods (non-overlapping {WINDOW_DAYS}-day blocks, MAPE ≤ {MAPE_THRESHOLD}%)"
# ax.set_title(title, fontsize=15, pad=20)

ax.set_ylabel("Price (USDT)", fontsize=20)

# ── Date (x) and Price (y) ticks ───────────────────────────────
ax.tick_params(axis='x', labelsize=18)          # Dates ── bigger
ax.tick_params(axis='y', labelsize=20)          # Prices ── even bigger

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=18))
plt.xticks(rotation=40, ha='right')

ax.grid(True, alpha=0.13, linestyle=':', zorder=0)

# Legend
legend_elements = [
    Patch(facecolor='none', edgecolor='#00c853', label='Real Bullish'),
    Patch(facecolor='none', edgecolor='#d50000', label='Real Bearish'),
]

if HAS_FULL_PRED:
    legend_elements += [
        Patch(facecolor='#ffca28', edgecolor='black', label='Predicted Bullish'),
        Patch(facecolor='black',   edgecolor='black', label='Predicted Bearish'),
        Patch(facecolor='gray', alpha=HIGHLIGHT_ALPHA, label=f'Positive Correlation'),
    ]

ax.legend(handles=legend_elements, loc='upper left',
          fontsize=22, framealpha=0.94)

plt.tight_layout()

# ── SAVE at 600 DPI ────────────────────────────────────────────
fig.savefig(OUTPUT_FILENAME, dpi=600, bbox_inches='tight', format='png')
print(f"Plot saved at 600 DPI: {OUTPUT_FILENAME}")

# Uncomment if you also want to display on screen
# plt.show()

# Optional: close figure to free memory (good when processing many coins)
# plt.close(fig)


In [ ]:
import os
import pandas as pd
from datetime import datetime

# ================= CONFIG =================
DATA_ROOT = "/home/nckh2/qa/finance/binance_ohlcv_daily"

coins = {
    'ADAUSDT':  f'{DATA_ROOT}/ADAUSDT_1d_full.csv',
    'AVAXUSDT': f'{DATA_ROOT}/AVAXUSDT_1d_full.csv',
    'BNBUSDT':  f'{DATA_ROOT}/BNBUSDT_1d_full.csv',
    'BTCUSDT':  f'{DATA_ROOT}/BTCUSDT_1d_full.csv',
    'DOGEUSDT': f'{DATA_ROOT}/DOGEUSDT_1d_full.csv',
    'ETHUSDT':  f'{DATA_ROOT}/ETHUSDT_1d_full.csv',
    'SOLUSDT':  f'{DATA_ROOT}/SOLUSDT_1d_full.csv',
    'LINKUSDT': f'{DATA_ROOT}/LINKUSDT_1d_full.csv',
    'TRXUSDT':  f'{DATA_ROOT}/TRXUSDT_1d_full.csv',
    'XRPUSDT':  f'{DATA_ROOT}/XRPUSDT_1d_full.csv',
}

# ================= SUMMARY FUNCTION =================
def summarize_coin(filepath):
    if not os.path.exists(filepath):
        return {
            'symbol': os.path.basename(filepath).replace('_1d_full.csv', ''),
            'status': 'File not found'
        }
    
    try:
        df = pd.read_csv(filepath, index_col='timestamp', parse_dates=True)
        df = df.sort_index()  # ensure chronological order
        
        if df.empty:
            return {
                'symbol': os.path.basename(filepath).replace('_1d_full.csv', ''),
                'status': 'Empty dataframe'
            }
        
        start_date = df.index.min().strftime('%Y-%m-%d')
        end_date   = df.index.max().strftime('%Y-%m-%d')
        total_obs  = len(df)
        
        # Expected number of days (from start to end inclusive)
        expected_days = (df.index.max() - df.index.min()).days + 1
        
        # Actual days present
        actual_days = total_obs
        
        # Number of missing days (gaps)
        missing_days = expected_days - actual_days
        
        # Basic close price stats
        close_min = df['close'].min()
        close_max = df['close'].max()
        
        # Completeness
        completeness_pct = (actual_days / expected_days * 100) if expected_days > 0 else 0
        
        return {
            'symbol': os.path.basename(filepath).replace('_1d_full.csv', ''),
            'start_date': start_date,
            'end_date': end_date,
            'total_obs': total_obs,
            'missing_days': missing_days,
            'completeness_%': round(completeness_pct, 2),
            'close_min': round(close_min, 4),
            'close_max': round(close_max, 4),
            'status': 'OK'
        }
    
    except Exception as e:
        return {
            'symbol': os.path.basename(filepath).replace('_1d_full.csv', ''),
            'status': f'Error: {str(e)}'
        }


# ================= MAIN =================
print(f"{'Symbol':<10} {'Start':<12} {'End':<12} {'Obs':>6} {'Missing':>8} {'Complete%':>10} {'Close Min':>12} {'Close Max':>12} {'Status'}")
print("-" * 90)

results = []

for symbol, path in coins.items():
    summary = summarize_coin(path)
    results.append(summary)
    
    if summary['status'] == 'OK':
        print(f"{summary['symbol']:<10} "
              f"{summary['start_date']:<12} "
              f"{summary['end_date']:<12} "
              f"{summary['total_obs']:>6,} "
              f"{summary['missing_days']:>8} "
              f"{summary['completeness_%']:>10.2f} "
              f"{summary['close_min']:>12,.4f} "
              f"{summary['close_max']:>12,.4f} "
              f"{summary['status']}")
    else:
        print(f"{summary['symbol']:<10} {'-':<12} {'-':<12} {'-':>6} {'-':>8} {'-':>10} {'-':>12} {'-':>12} {summary['status']}")

print("\nSummary complete.")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─── Configuration ───────────────────────────────────────────────────────
DATA_PATH = "/home/nckh2/qa/ultimate.csv"
OUTPUT_DIR = "/home/nckh2/qa/plots"   # ← or "/home/nckh2/qa/plots", etc.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Plot Settings ───────────────────────────────────────────────────────
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 15,
    'axes.titlesize': 19,
    'axes.labelsize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 15,
    'figure.dpi': 100,
    'savefig.dpi': 600,
})

FIGSIZE = (10, 6)

# ─── Colors (added new models) ───────────────────────────────────────────
MODEL_COLORS = {
    'OmniFormer':         '#D81E5B',   # your strong performer (if it appears later)
    'EnFormer':     '#FF6B6B',   # vivid coral/reddish — new top model
    'iTransformer': '#1E90FF',   # bright blue
    'PatchTST':     '#32CD32',   # lime green
    'Autoformer':   '#FFECB3',   # pale yellow
    'FEDformer':    '#C4F4E3',   # very light green
    'Informer':     '#FFD166',   # mustard/orange
    'MoLE':         '#F4A261',   # orange (if still present)
    'Linear':       '#A3BFFA',
    'DLinear':      '#B3E5FC',
    'NLinear':      '#81D4FA',
    'RLinear':      '#4FC3F7',
    'RNN':          '#999999',   # gray — weak
    'LSTM':         '#777777',
    'GRU':          '#555555',
    'CLAM':         '#333333',
    'Default':      '#A9A9A9'
}

# Extended reference order — roughly by expected strength
REFERENCE_ORDER = [
    'EnFormer', 'iTransformer', 'PatchTST', 'HIEU',
    'Autoformer', 'FEDformer', 'Informer',
    'MoLE', 'Linear', 'DLinear', 'NLinear', 'RLinear',
    'RNN', 'LSTM', 'GRU', 'CLAM'
]

# Baselines for HIEU comparison (non-HIEU, non-recurrent if you want — but kept broad)
BASELINES_ORDER = [
    'EnFormer', 'iTransformer', 'PatchTST',
    'Autoformer', 'FEDformer', 'Informer',
    'RNN', 'LSTM', 'GRU', 'CLAM'
]

# ─── Load & Prepare ──────────────────────────────────────────────────────
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"File not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
# Rename columns to match old code expectation
df = df.rename(columns={'coin': 'asset', 'mae_scaled': 'MAE_mean'})

print("Loaded data — columns:", df.columns.tolist())
print("Models:", sorted(df['model'].unique().tolist()))
print("Assets:", sorted(df['asset'].unique().tolist()))

required = {'model', 'asset', 'MAE_mean'}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")

# Optional: filter only test set or specific subset if your CSV has extra rows
# df = df[df['split'] == 'test']   # ← uncomment/adapt if needed

# ─── 1. Average Rank Bar Plot ────────────────────────────────────────────
def plot_average_rank(df, out_dir):
    df_rank = df[df['asset'] != 'Average'].copy() if 'Average' in df['asset'].values else df.copy()
    
    df_rank['rank'] = df_rank.groupby('asset')['MAE_mean'].rank(method='average', ascending=True)
    
    rank_stats = df_rank.groupby('model')['rank'].agg(['mean', 'std']).rename(
        columns={'mean': 'avg_rank', 'std': 'std_rank'}
    ).sort_values('avg_rank')
    
    colors = [MODEL_COLORS.get(m, MODEL_COLORS['Default']) for m in rank_stats.index]
    
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.bar(rank_stats.index, rank_stats['avg_rank'], yerr=rank_stats['std_rank'],
           capsize=5, color=colors, edgecolor='black', linewidth=0.8)
    
    ax.set_ylabel('Average Rank')
    ax.set_xlabel('Model')
    # ax.set_title('Average Model Rank Across Assets')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    plt.tight_layout()
    path = os.path.join(out_dir, 'barplot_average_rank.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f"Saved: {path}")

# ─── 2. Normalized MAE Line Plot (top 5) ─────────────────────────────────
def plot_normalized_mae(df, out_dir):
    asset_order = sorted(df['asset'].unique())   # or keep your old volatility order if desired
    
    df_temp = df[df['asset'] != 'Average'].copy()
    df_temp['rank'] = df_temp.groupby('asset')['MAE_mean'].rank(method='average', ascending=True)
    avg_ranks = df_temp.groupby('model')['rank'].mean().sort_values()
    top_n = avg_ranks.head(5).index.tolist()
    
    print("Top 5 models:", top_n)
    
    df_norm = df[df['model'].isin(top_n) & (df['asset'] != 'Average')].copy()
    df_norm['mean_asset_mae'] = df_norm.groupby('asset')['MAE_mean'].transform('mean')
    df_norm['norm_mae'] = df_norm['MAE_mean'] / df_norm['mean_asset_mae']
    
    df_norm['asset'] = pd.Categorical(df_norm['asset'], categories=asset_order, ordered=True)
    df_norm = df_norm.sort_values('asset')
    
    fig, ax = plt.subplots(figsize=FIGSIZE)
    for model in top_n:
        sub = df_norm[df_norm['model'] == model]
        ax.plot(sub['asset'], sub['norm_mae'], marker='o', linewidth=2.3, markersize=8,
                label=model, color=MODEL_COLORS.get(model, MODEL_COLORS['Default']))
    
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.4, alpha=0.7)
    ax.set_xlabel('Asset')
    ax.set_ylabel('Normalized MAE')
    # ax.set_title('Relative Performance — Top 5 Models')
    ax.legend(frameon=True, edgecolor='gray', loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, linestyle='--', alpha=0.35)
    
    plt.tight_layout()
    path = os.path.join(out_dir, 'lineplot_normalized_mae_top5.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f"Saved: {path}")

# ─── 3. Relative Improvement Boxplot (if HIEU present) ───────────────────
def plot_relative_improvement(df, out_dir):
    if 'HIEU' not in df['model'].unique():
        print("No 'HIEU' found → skipping relative improvement boxplot")
        return
    
    df_rel = df[df['asset'] != 'Average'].copy()
    hieu_mae = df_rel[df_rel['model'] == 'HIEU'].set_index('asset')['MAE_mean']
    
    records = []
    for base in BASELINES_ORDER:
        if base not in df_rel['model'].unique():
            continue
        base_mae = df_rel[df_rel['model'] == base].set_index('asset')['MAE_mean']
        common = hieu_mae.index.intersection(base_mae.index)
        if len(common) == 0:
            continue
        rel = (base_mae[common] - hieu_mae[common]) / base_mae[common]
        for v in rel:
            records.append({'baseline': base, 'rel_improvement': v})
    
    if not records:
        print("No comparable baselines → skipping boxplot")
        return
    
    df_plot = pd.DataFrame(records)
    
    fig, ax = plt.subplots(figsize=FIGSIZE)
    labels = [m for m in BASELINES_ORDER if m in df_plot['baseline'].unique()]
    data = [df_plot[df_plot['baseline'] == m]['rel_improvement'] for m in labels]
    
    box = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=True, widths=0.22)
    
    for patch, model in zip(box['boxes'], labels):
        patch.set_facecolor(MODEL_COLORS.get(model, MODEL_COLORS['Default']))
        patch.set_alpha(0.85)
    
    ax.axhline(0, color='crimson', linestyle='--', linewidth=1.5)
    ax.set_ylabel('Relative Improvement over Baseline\n((base - model)/base)')
    ax.set_xlabel('Baseline Model')
    ax.set_title('Relative Improvement Distribution')
    ax.tick_params(axis='x', rotation=40)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    
    plt.tight_layout()
    path = os.path.join(out_dir, 'boxplot_relative_improvement.png')
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print(f"Saved: {path}")

# ─── Run ─────────────────────────────────────────────────────────────────
plot_average_rank(df, OUTPUT_DIR)
plot_normalized_mae(df, OUTPUT_DIR)
plot_relative_improvement(df, OUTPUT_DIR)

print("\nAll done! Plots saved to:", OUTPUT_DIR)

In [ ]:
import pandas as pd
import os

# ─── Configuration ───────────────────────────────────────────────────────
CSV_PATH      = "/home/nckh2/qa/ultimate.csv"
TARGET_MODEL  = "OmniFormer"          # ← CHANGE THIS to your best/new model name
                                    #   e.g. "EnFormer", "OmniFormer", "HIEU", etc.

METRIC_COL    = "mae_scaled"        # we use this as the error measure

# ─── Load & Prepare ──────────────────────────────────────────────────────
if not os.path.isfile(CSV_PATH):
    print(f"File not found: {CSV_PATH}")
    exit(1)

df = pd.read_csv(CSV_PATH)

# Basic sanity check
expected_cols = {'model', 'coin', METRIC_COL}
if not expected_cols.issubset(df.columns):
    print("Missing required columns. Found:", df.columns.tolist())
    exit(1)

# Clean up: remove any summary rows if they exist
df = df[df['coin'] != 'Average'].copy()   # adjust if your average row uses different label

print("Models found:", sorted(df['model'].unique().tolist()))
print(f"Assets/coins: {sorted(df['coin'].unique().tolist())}")
print(f"Rows: {len(df)}")

# ─── Compute relative improvements of TARGET_MODEL over others ───────────
def compute_relative_improvements(df, target_model, metric_col="mae_scaled"):
    if target_model not in df['model'].unique():
        print(f"\nError: Target model '{target_model}' not found.")
        print("Available models:", sorted(df['model'].unique()))
        return None, None, None

    target_error = df[df['model'] == target_model].set_index('coin')[metric_col]

    results = []
    all_improvements = []  # for grand average

    other_models = [m for m in df['model'].unique() if m != target_model]

    for base_model in sorted(other_models):
        base_error = df[df['model'] == base_model].set_index('coin')[metric_col]

        # Only assets present in both target and baseline
        common_coins = target_error.index.intersection(base_error.index)
        if len(common_coins) == 0:
            continue

        target_vals = target_error[common_coins]
        base_vals   = base_error[common_coins]

        # Relative improvement in % (positive = target is better)
        rel_improvement_pct = (base_vals - target_vals) / base_vals * 100

        avg_pct     = rel_improvement_pct.mean()
        count       = len(rel_improvement_pct)
        min_pct     = rel_improvement_pct.min()
        max_pct     = rel_improvement_pct.max()

        results.append({
            'baseline':       base_model,
            'avg_improvement_%': round(avg_pct, 2),
            'assets':         count,
            'worst_%':        round(min_pct, 2),
            'best_%':         round(max_pct, 2),
        })

        all_improvements.extend(rel_improvement_pct)

    if not results:
        print("No common asset comparisons found.")
        return None, None, None

    df_results = pd.DataFrame(results).sort_values('avg_improvement_%', ascending=False)

    overall_avg_pct = round(pd.Series(all_improvements).mean(), 2) if all_improvements else None
    overall_n       = len(all_improvements)

    return df_results, overall_avg_pct, overall_n


# ─── Run ─────────────────────────────────────────────────────────────────
df_improv, overall_avg, overall_count = compute_relative_improvements(
    df, TARGET_MODEL, METRIC_COL
)

if df_improv is not None:
    print(f"\nRelative improvement of **{TARGET_MODEL}** vs other models "
          f"(based on mae_scaled — positive = {TARGET_MODEL} better)\n")

    # Nice table output
    print(df_improv.to_string(index=False))
    print()

    if overall_avg is not None:
        print(f"Overall: {TARGET_MODEL} is {overall_avg:+.2f}% better on average "
              f"across {overall_count} direct (model-asset) comparisons.")
    else:
        print("No valid overall average could be computed.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─── Configuration ───────────────────────────────────────────────────────
CSV_PATH   = "/home/nckh2/qa/ultimate.csv"
OUTPUT_DIR = "/home/nckh2/qa/plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(OUTPUT_DIR, "barplot_mae_rmse_per_model.png")

MAIN_MODEL = "OmniFormer"

# ─── Global Plot Settings (same as your second script) ───────────────────
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 15,
    'axes.titlesize': 19,
    'axes.labelsize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 15,
    'figure.dpi': 100,
    'savefig.dpi': 600,
})

FIGSIZE = (10, 6)

# ─── Same MODEL_COLORS dictionary from your code ─────────────────────────
MODEL_COLORS = {
    'OmniFormer': '#D81E5B',
    'EnFormer': '#FF6B6B',
    'iTransformer': '#1E90FF',
    'PatchTST': '#32CD32',
    'Autoformer': '#FFECB3',
    'FEDformer': '#C4F4E3',
    'Informer': '#FFD166',
    'MoLE': '#F4A261',
    'Linear': '#A3BFFA',
    'DLinear': '#B3E5FC',
    'NLinear': '#81D4FA',
    'RLinear': '#4FC3F7',
    'RNN': '#999999',
    'LSTM': '#777777',
    'GRU': '#555555',
    'CLAM': '#333333',
    'Default': '#A9A9A9'
}

# ─── Choose 2 colors from your palette ───────────────────────────────────
COLOR_MAE  = MODEL_COLORS['EnFormer']
COLOR_RMSE = MODEL_COLORS['iTransformer']

# ─── Load & Aggregate ─────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

agg = df.groupby('model')[['mae_scaled', 'rmse_scaled']].mean().reset_index()

others = agg[agg['model'] != MAIN_MODEL].sort_values('mae_scaled')

if MAIN_MODEL in agg['model'].values:
    main_row = agg[agg['model'] == MAIN_MODEL]
    agg_sorted = pd.concat([main_row, others], ignore_index=True)
else:
    agg_sorted = agg.sort_values('mae_scaled').reset_index(drop=True)
    MAIN_MODEL = agg_sorted['model'].iloc[0]

# ─── Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=FIGSIZE)

models = agg_sorted['model']
x = np.arange(len(models))
width = 0.35

rects1 = ax.bar(
    x - width/2,
    agg_sorted['mae_scaled'],
    width,
    label='MAE',
    color=COLOR_MAE,
    edgecolor='black',
    linewidth=0.8
)

rects2 = ax.bar(
    x + width/2,
    agg_sorted['rmse_scaled'],
    width,
    label='RMSE',
    color=COLOR_RMSE,
    edgecolor='black',
    linewidth=0.8
)

# ─── Styling ──────────────────────────────────────────────────────────────
ax.set_xlabel('Model')
ax.set_ylabel('Error')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')

ax.legend(frameon=True, edgecolor='gray')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)

# Value labels
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width()/2, height),
                    xytext=(0, 4),
                    textcoords="offset points",
                    ha='center',
                    va='bottom',
                    fontsize=10)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.savefig(SAVE_PATH, bbox_inches='tight')
plt.close()

print(f"Bar plot saved to: {SAVE_PATH}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from IPython.display import display, Markdown
import os

# ────────────────────────────────────────────────
#                 CONFIGURATION
# ────────────────────────────────────────────────

BASE_PATH = "/home/nckh2/qa/finance/binance_ohlcv_daily/"

FORECAST_FILES = {
    "OmniFormer": "/home/nckh2/qa/forecasts_EnFormer_refined.csv"
    # Add more models here later if needed
}

MODEL_COLORS = {
    "OmniFormer": "red",
    # Add more model colors here if needed
}

COINS = [
    "ADA", "AVAX", "BNB", "BTC", "DOGE",
    "ETH", "SOL", "LINK", "TRX", "XRP"
]

# Where to save the plots
SAVE_DIR = "/home/nckh2/qa/finance/plots/forecast"
os.makedirs(SAVE_DIR, exist_ok=True)  # Create folder if it doesn't exist


# ────────────────────────────────────────────────
#                  DATA LOADING
# ────────────────────────────────────────────────

def load_historical(coin):
    file_path = f"{BASE_PATH}{coin.upper()}USDT_1d_full.csv"
    try:
        hist = pd.read_csv(file_path)
        hist['timestamp'] = pd.to_datetime(hist['timestamp'])
        hist = hist.rename(columns={'close': 'close_real'})
        # We only use data before forecast start → no future leakage
        return hist[['timestamp', 'close_real']].sort_values('timestamp').reset_index(drop=True)
    except FileNotFoundError:
        return None


def load_forecast(model, coin):
    if model not in FORECAST_FILES:
        return None
    forecast_path = FORECAST_FILES[model]
    try:
        fc = pd.read_csv(forecast_path)
        close_pred_col = f"{coin.upper()}USDT_close_one_step"
        if close_pred_col not in fc.columns:
            return None
        fc = pd.DataFrame({
            'timestamp': pd.to_datetime(fc['timestamp']),
            'close_pred': fc[close_pred_col]
        })
        return fc
    except FileNotFoundError:
        return pd.DataFrame()  # empty


# ────────────────────────────────────────────────
#              PLOTTING FUNCTION
# ────────────────────────────────────────────────

def plot_full_history_plus_predictions(coin, models=None, figsize=(7, 6), zoom_last_days=None):
    if models is None:
        models = list(FORECAST_FILES.keys())

    hist = load_historical(coin)
    if hist is None:
        display(Markdown(f"**{coin}**: historical data not found"))
        return

    fig, ax = plt.subplots(figsize=figsize, dpi=100)  # screen dpi irrelevant for saving

    first_forecast_date = None

    # Plot predictions
    for model in models:
        fc = load_forecast(model, coin)
        if fc is None or fc.empty:
            continue

        if first_forecast_date is None:
            first_forecast_date = fc["timestamp"].min()
            ax.axvline(first_forecast_date, color="0.5", linestyle=":", linewidth=1.6,
                       label="Forecast start", zorder=5)

        color = MODEL_COLORS.get(model, "#555555")
        ax.plot(fc["timestamp"], fc["close_pred"],
                color=color, linewidth=1.9, linestyle="--", alpha=0.92,
                label=f"{model} prediction", zorder=3)

    if first_forecast_date is None:
        display(Markdown(f"**{coin}**: no forecast data found for selected models"))
        plt.close(fig)
        return

    # Plot only historical data BEFORE forecast start
    hist_before = hist[hist['timestamp'] < first_forecast_date]
    ax.plot(hist_before["timestamp"], hist_before["close_real"],
            color="royalblue", linewidth=1.35,
            label="Historical close", zorder=10)

    # ── Styling ───────────────────────────────────────────────────
    ax.set_title(f"{coin}USDT", fontsize=21, pad=18)                    # was 14 → ~21
    ax.set_ylabel("Close Price (USDT)", fontsize=18)                    # was 12 → 18
    ax.set_xlabel("Date", fontsize=18)                                  # was 12 → 18

    ax.set_xlim(left=datetime(2021, 1, 1))

    if zoom_last_days is not None:
        zoom_from = max(datetime(2021, 1, 1),
                        hist["timestamp"].max() - pd.Timedelta(days=zoom_last_days + 60))
        ax.set_xlim(zoom_from, hist["timestamp"].max() + pd.Timedelta(days=20))

    ax.legend(loc="upper left", fontsize=11, ncol=2, framealpha=0.92)  # was 9.8 → ~15

    ax.grid(True, alpha=0.22, linestyle="--")

    # Tick labels ~1.5× default size
    ax.tick_params(axis='both', which='major', labelsize=15)            # default ~10–11 → 15

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    plt.xticks(rotation=35)

    # ax.set_yscale('log')   # uncomment if you prefer log scale

    plt.tight_layout()

    # ── SAVE at 600 DPI ───────────────────────────────────────────
    filename = os.path.join(SAVE_DIR, f"{coin}USDT_forecast.png")
    fig.savefig(filename, dpi=600, bbox_inches='tight', format='png')
    print(f"Saved: {filename}")

    # Uncomment if you still want to display in notebook
    # plt.show()

    plt.close(fig)  # crucial when looping many coins


# ────────────────────────────────────────────────
#                   RUN ALL
# ────────────────────────────────────────────────

print(f"Saving all plots to: {SAVE_DIR}\n")

for coin in COINS:
    plot_full_history_plus_predictions(coin)
    # If you want recent zoom on all charts, use:
    # plot_full_history_plus_predictions(coin, zoom_last_days=730)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─── Output Path ──────────────────────────────────────────────────────────
OUTPUT_DIR = "/home/nckh2/qa/plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SAVE_PATH = os.path.join(OUTPUT_DIR, "computational_overhead_hybrid.png")

# ─── Global Plot Settings (EXACT same as your script) ─────────────────────
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 15,
    'axes.titlesize': 19,
    'axes.labelsize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 15,
    'figure.dpi': 100,
    'savefig.dpi': 600,
})

FIGSIZE = (10, 6)

# ─── Same MODEL_COLORS dictionary ─────────────────────────────────────────
MODEL_COLORS = {
    'OmniFormer': '#D81E5B',
    'EnFormer': '#FF6B6B',
    'iTransformer': '#1E90FF',
    'PatchTST': '#32CD32',
    'Autoformer': '#FFECB3',
    'FEDformer': '#C4F4E3',
    'Informer': '#FFD166',
    'MoLE': '#F4A261',
    'Linear': '#A3BFFA',
    'DLinear': '#B3E5FC',
    'NLinear': '#81D4FA',
    'RLinear': '#4FC3F7',
    'RNN': '#999999',
    'LSTM': '#777777',
    'GRU': '#555555',
    'CLAM': '#333333',
    'Default': '#A9A9A9'
}

# ─── Use same color logic ─────────────────────────────────────────────────
COLOR_TRAIN = MODEL_COLORS['EnFormer']
COLOR_INFER = MODEL_COLORS['iTransformer']
COLOR_RUNTIME = MODEL_COLORS['OmniFormer']

# ─── Data (excluding runtime for bars) ────────────────────────────────────
metrics = [
    "CO2 (kg)",
    "Energy (kWh)",
    "CPU Power (W)",
    "GPU Power (W)",
    "CPU Util (%)",
    "GPU Util (%)"
]

training_bar = [0.212, 0.446, 41.840, 206.648, 5.562, 93.061]
inference_bar = [0.000, 0.000, 41.753, 174.373, 0.000, 85.000]

runtime_training = 6067.814
runtime_inference = 3.061

# ─── Plot ─────────────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=FIGSIZE)

x = np.arange(len(metrics))
width = 0.35

# Bars
bars1 = ax1.bar(
    x - width/2,
    training_bar,
    width,
    label='Training',
    color=COLOR_TRAIN,
    edgecolor='black',
    linewidth=0.8
)

bars2 = ax1.bar(
    x + width/2,
    inference_bar,
    width,
    label='Inference',
    color=COLOR_INFER,
    edgecolor='black',
    linewidth=0.8
)

# Secondary axis for runtime
ax2 = ax1.twinx()

# Training runtime (same color as training bars)
ax2.plot(
    [-0.5, len(metrics)-0.5],
    [runtime_training, runtime_training],
    linestyle='--',
    marker='o',
    linewidth=2,
    color=COLOR_TRAIN,
    label='Training Runtime (s)'
)

# Inference runtime (same color as inference bars)
ax2.plot(
    [-0.5, len(metrics)-0.5],
    [runtime_inference, runtime_inference],
    linestyle='--',
    marker='s',
    linewidth=2,
    color=COLOR_INFER,
    alpha=0.9,
    label='Inference Runtime (s)'
)


# ─── Styling ──────────────────────────────────────────────────────────────
ax1.set_xlabel("Metric")
ax1.set_ylabel("Metric Value")
ax2.set_ylabel("Runtime (s)")

ax1.set_xticks(x)
ax1.set_xticklabels(metrics, rotation=45, ha='right')

ax1.grid(axis='y', linestyle='--', alpha=0.35)
ax1.set_axisbelow(True)

# Combined legend
# Combined legend (CENTER LEFT)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc='center left',
    bbox_to_anchor=(0.02, 0.5),   # slightly inside the plot
    frameon=True,
    edgecolor='gray'
)

# plt.title("Computational Overhead of OmniFormer")

plt.tight_layout()
plt.savefig(SAVE_PATH, bbox_inches='tight')
plt.close()

print(f"Hybrid plot saved to: {SAVE_PATH}")